<a href="https://colab.research.google.com/github/JoseHdez210201/IanFest_Semaforo_ESP32/blob/main/FOMO_TALLER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/TALLER'

print("Google Drive conectado correctamente.")

Mounted at /content/drive
Google Drive conectado correctamente.


In [ ]:
import tensorflow as tf
import numpy as np
import os

# Modelo de detección de rostros
ruta_modelo = os.path.join(
    CARPETA,
    "ei-face-detection---fomo-object-detection-tensorflow-lite-int8-quantized-model.19.lite"
)

# Cargar modelo
interpreter = tf.lite.Interpreter(model_path=ruta_modelo)
interpreter.allocate_tensors()

# Información de entrada y salida
entrada = interpreter.get_input_details()[0]
salida = interpreter.get_output_details()[0]

print("Modelo FOMO cargado correctamente.")
print("Dimensiones de entrada:", entrada['shape'])
print("Dimensiones de salida:", salida['shape'])
print("Tipo de entrada:", entrada['dtype'])
print("Tipo de salida:", salida['dtype'])
print("Cuantización de entrada:", entrada['quantization'])
print("Cuantización de salida:", salida['quantization'])

Modelo FOMO cargado correctamente.
Dimensiones de entrada: [ 1 96 96  3]
Dimensiones de salida: [ 1 12 12  2]
Tipo de entrada: <class 'numpy.int8'>
Tipo de salida: <class 'numpy.int8'>
Cuantización de entrada: (0.003921568859368563, -128)
Cuantización de salida: (0.00390625, -128)


In [ ]:

import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
import os
import io
import time

from PIL import Image, ImageOps
from scipy.ndimage import label
from IPython.display import display, clear_output
from google.colab import files

# Obtener dimensiones automáticamente
_, ALTO, ANCHO, CANALES = entrada['shape']

# ==========================================
# FUNCIÓN DE DETECCIÓN FOMO
# ==========================================

def detectar_rostros(imagen):

    imagen = ImageOps.exif_transpose(imagen).convert('RGB')

    # 1. Preparar imagen de entrada
    modo = 'L' if CANALES == 1 else 'RGB'

    img = imagen.convert(modo).resize(
        (ANCHO, ALTO),
        Image.Resampling.BILINEAR
    )

    # 2. Normalizar los píxeles
    datos = np.array(img, dtype=np.float32) / 255.0

    if CANALES == 1:
        datos = datos[..., np.newaxis]

    # 3. Cuantizar entrada a Int8
    escala, cero = entrada['quantization']
    limites = np.iinfo(entrada['dtype'])

    datos = np.clip(
        np.round(datos / escala + cero),
        limites.min,
        limites.max
    ).astype(entrada['dtype'])

    datos = np.expand_dims(datos, axis=0)

    # 4. Ejecutar inferencia
    interpreter.set_tensor(entrada['index'], datos)

    inicio = time.perf_counter()
    interpreter.invoke()
    tiempo = (time.perf_counter() - inicio) * 1000

    resultado = interpreter.get_tensor(salida['index'])

    # 5. Descuantizar probabilidades
    escala, cero = salida['quantization']

    resultado = (
        resultado.astype(np.float32) - cero
    ) * escala

    resultado = resultado[0]

    # Clase 0: fondo
    # Clase 1: rostro
    mapa = resultado[:, :, 1]

    filas, columnas = mapa.shape

    # 6. Detectar regiones con rostros
    mascara = (
        (mapa >= umbral.value) &
        (np.argmax(resultado, axis=-1) == 1)
    )

    regiones, cantidad = label(
        mascara,
        structure=np.ones((3, 3))
    )

    # 7. Preparar visualización
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))

    ax[0].imshow(imagen)
    ax[0].axis('off')

    print("=" * 45)
    print("RESULTADOS DE LA INFERENCIA")
    print("=" * 45)
    print(f"Modelo: FOMO Int8")
    print(f"Entrada: {ANCHO} x {ALTO} px")
    print(f"Cuadrícula: {columnas} x {filas}")
    print(f"Inferencia en Colab: {tiempo:.2f} ms")
    print(f"Rostros detectados: {cantidad}")
    print("=" * 45)

    if cantidad > 0:

        ax[0].set_title(
            f"Rostros detectados: {cantidad}",
            color="green",
            fontsize=16,
            fontweight="bold"
        )

        for i in range(1, cantidad + 1):

            ys, xs = np.where(regiones == i)

            # Calcular centroide
            x = (xs.mean() + 0.5) / columnas
            y = (ys.mean() + 0.5) / filas

            x *= imagen.width
            y *= imagen.height

            confianza = mapa[ys, xs].max()

            # Dibujar centroide
            ax[0].scatter(
                x, y,
                c="lime",
                s=180,
                edgecolors="white",
                linewidths=2
            )

            # Mostrar etiqueta y confianza
            ax[0].annotate(
                f"Rostro {i}: {confianza:.1%}",
                (x, y),
                xytext=(8, -15),
                textcoords="offset points",
                color="white",
                fontsize=11,
                bbox=dict(
                    facecolor="green",
                    alpha=0.85,
                    edgecolor="none"
                )
            )

            print(f"Rostro {i}")
            print(f"Centroide: ({x:.0f}, {y:.0f})")
            print(f"Confianza: {confianza:.1%}")
            print()

    else:

        print("No se detectaron rostros.")

        ax[0].set_title(
            "No se detectaron rostros",
            color="red",
            fontsize=16,
            fontweight="bold"
        )

    # 8. Visualizar mapa de calor
    calor = ax[1].imshow(
        mapa,
        cmap="jet",
        vmin=0,
        vmax=1,
        interpolation="nearest"
    )

    ax[1].set_title(
        f"Mapa de calor FOMO ({columnas} x {filas})",
        fontsize=14,
        fontweight="bold"
    )

    ax[1].set_xticks(range(columnas))
    ax[1].set_yticks(range(filas))

    fig.colorbar(
        calor,
        ax=ax[1],
        fraction=0.046,
        label="Probabilidad"
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ==========================================
# INTERFAZ INTERACTIVA
# ==========================================

imagenes = sorted([
    nombre for nombre in os.listdir(CARPETA)
    if nombre.lower().endswith(
        ('.jpg', '.jpeg', '.png')
    )
])

selector = widgets.Dropdown(
    options=imagenes,
    description="Imagen:",
    layout=widgets.Layout(width="400px")
)

umbral = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=0.99,
    step=0.01,
    description="Umbral:",
    readout_format=".2f"
)

boton_detectar = widgets.Button(
    description="Detectar rostros",
    button_style="success",
    icon="search"
)

boton_subir = widgets.Button(
    description="Subir nueva imagen",
    button_style="info",
    icon="upload"
)

output = widgets.Output()

# Detectar imágenes de Drive
def ejecutar(b):
    with output:
        clear_output(wait=True)

        ruta = os.path.join(CARPETA, selector.value)

        detectar_rostros(Image.open(ruta))

# Subir imágenes desde la computadora
def subir_imagen(b):
    with output:
        clear_output(wait=True)

        archivos = files.upload()

        for nombre, contenido in archivos.items():
            print("Imagen:", nombre)

            imagen = Image.open(io.BytesIO(contenido))

            detectar_rostros(imagen)

boton_detectar.on_click(ejecutar)
boton_subir.on_click(subir_imagen)

# Mostrar interfaz
display(widgets.HTML(
    """
    <h2 style="color:#1A4B8E;">
    Demostración TinyML - FOMO
    </h2>
    <p>
    Detección de rostros mediante centroides<br>
    Modelo MobileNetV2 - Cuantización Int8
    </p>
    """
))

display(widgets.VBox([
    selector,
    umbral,
    widgets.HBox([
        boton_detectar,
        boton_subir
    ]),
    output
]))


HTML(value='\n    <h2 style="color:#1A4B8E;">\n    Demostración TinyML - FOMO\n    </h2>\n    <p>\n    Detecci…